In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

root = Path.cwd()
RESULTS = root / 'results'
DATA = root / 'data'

signals = pd.read_csv(RESULTS / 'signals_ALL_pairs.csv')
signals['date'] = pd.to_datetime(signals['date'])

kalman = pd.read_csv(RESULTS / 'kalman_dynamic_spreads.csv')
kalman['date'] = pd.to_datetime(kalman['date'])
kalman['pair'] = kalman['stock1'].astype(str) + '_' + kalman['stock2'].astype(str)

master = signals.merge(kalman[['pair', 'date', 'beta_kf', 'spread_kf']], on=['pair', 'date'], how='inner')
master = master.sort_values(['pair', 'date']).reset_index(drop=True)

print("Merged:", master.shape)
print(master[['pair', 'date', 'signal', 'position', 'size', 'beta_kf', 'spread_kf']].head())


Merged: (58567, 12)
       pair       date  signal  position  size   beta_kf  spread_kf
0  ABNB_APD 2021-06-04       0       0.0   0.0  0.480293   0.015783
1  ABNB_APD 2021-06-07       0       0.0   0.0  0.480482   0.007996
2  ABNB_APD 2021-06-08       0       0.0   0.0  0.480453  -0.001606
3  ABNB_APD 2021-06-09       0       0.0   0.0  0.480301  -0.011715
4  ABNB_APD 2021-06-10       0       0.0   0.0  0.480228  -0.004592


In [2]:


for col in ['position', 'size', 'beta_kf']:
    master[f'{col}_t1'] = master.groupby('pair')[col].shift(1)

master['signal_persist'] = master.groupby('pair')['signal'].transform(
    lambda x: x.replace(0, np.nan).ffill().fillna(0)
)
master['position_t1'] = master.groupby('pair')['signal_persist'].shift(1)


master['spread_ret'] = master.groupby('pair')['spread_kf'].diff()
master['pnl_gross'] = -master['size_t1'] * master['spread_ret']


master['delta_size'] = master.groupby('pair')['size_t1'].diff().fillna(0)


master['slippage'] = master['delta_size'].abs() * 0.0010  


master['borrow'] = master['size_t1'].abs() * (0.0100 / 252)  


master['pnl_net'] = master['pnl_gross'] - master['slippage'] - master['borrow']

print("PnL after persistence:")
print(master[['pnl_gross', 'slippage', 'borrow', 'pnl_net']].describe())


PnL after persistence:
          pnl_gross      slippage        borrow       pnl_net
count  58548.000000  5.856700e+04  5.854800e+04  58548.000000
mean       0.000034  4.704014e-07  1.922009e-06      0.000032
std        0.000796  6.782965e-06  3.455344e-07      0.000795
min       -0.021863  0.000000e+00  0.000000e+00     -0.021865
25%       -0.000309  0.000000e+00  1.984127e-06     -0.000311
50%        0.000000  0.000000e+00  1.984127e-06      0.000000
75%        0.000359  0.000000e+00  1.984127e-06      0.000356
max        0.013875  1.000000e-04  1.984127e-06      0.013873


In [3]:
daily = master.dropna(subset='pnl_net').groupby('date')['pnl_net'].sum().sort_index()
equity = (1 + daily).cumprod()
peak = equity.expanding().max()
dd = (equity / peak - 1)

dd_breach = (dd.shift(1) < -0.20).fillna(False)
daily_capped = daily.where(~dd_breach, 0)
equity_capped = (1 + daily_capped).cumprod()

print("Uncapped max DD:", dd.min())
print("Capped max DD:", (equity_capped / equity_capped.expanding().max() - 1).min())
print("Breaches:", dd_breach.sum())



Uncapped max DD: -0.037646405421094764
Capped max DD: -0.037646405421094764
Breaches: 0


In [4]:

master['size_abs'] = master['size_t1'].abs()
master['delta_abs'] = master['delta_size'].abs()

gross_daily = master.groupby('date')['size_abs'].sum()
notional_daily = master.groupby('date')['delta_abs'].sum()
valid_days = gross_daily > 0
turnover_daily = notional_daily[valid_days] / gross_daily[valid_days]
turnover = turnover_daily.mean() * 252

ann_ret = daily_capped.mean() * 252
vol_ann = daily_capped.std() * np.sqrt(252)
sharpe = ann_ret / vol_ann if vol_ann > 0 else 0
max_dd = (equity_capped / equity_capped.expanding().max() - 1).min()
calmar = ann_ret / -max_dd if max_dd < 0 else np.inf
win_rate = (daily_capped > 0).mean()

print("=== Backtest complete ===")
print(f"Sharpe:      {sharpe:.2f}")
print(f"Total Return:{(equity_capped.iloc[-1]-1)*100:.1f}%")
print(f"Max DD:      {max_dd:.1%}")
print(f"Calmar:      {calmar:.2f}")
print(f"Turnover:    {turnover:.1f}x")
print(f"Win Rate:    {win_rate:.0%}")
print(f"Breaches:    {dd_breach.sum()}")





=== Backtest complete ===
Sharpe:      2.11
Total Return:534.0%
Max DD:      -3.8%
Calmar:      3.16
Turnover:    2.5x
Win Rate:    55%
Breaches:    0
